# Week 8, Day 1: Your pricer as a Modal service

**Lesson:** [week8/day1.ipynb](../../../week8/day1.ipynb), *Modal.com and the specialist agent*. **About 35 minutes.**

**What you practice here**

- Running a function in Modal's cloud from Python: `app.run()` and `.remote()`, with `modal.enable_output()` on so that you can watch it.
- Describing the container a GPU model needs: a `modal.Image` with its pip packages, the Hugging Face secret from `modal.Secret.from_name`, the GPU and the timeout.
- Calling a deployed Modal class from any Python with `modal.Cls.from_name`: one instance, then its method's `.remote`.
- Keeping a deployed service warm with `update_autoscaler`, and letting it cool down again.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week8/day1", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson puts the pricer you fine-tuned in Week 7 on Modal. It runs functions in Modal's cloud, describes the GPU
container that the model needs, deploys the pricer as a class, and calls that class from the notebook. Write that
Python for a desk that prices wrestling merchandise: an Elite-series Stone Cold figure, a replica Winged Eagle title
belt, a 1991 Hasbro Undertaker.

Each part takes the `modal` module as its first parameter and calls it exactly as you would after `import modal`. The
checks hand your parts a stand-in shaped like `modal`. It records what you look up and call, refuses a remote call on
an app that is not running, as Modal does, and answers with canned prices, so nothing is deployed and nothing is
spent. The service itself, the class that Modal deploys, goes in Try it for real: Modal reads it from a file of its own.

## What each call should give

```python
from hello import app, hello_europe      # the lesson's hello.py
run_remotely(modal, app, hello_europe)
# 'Hello from Frankfurt am Main, Hesse, DE!!'   (it ran in a Modal container in Europe)

service_options(modal)
# {'image': <a Debian slim image with torch, transformers, bitsandbytes, accelerate and peft installed>,
#  'secrets': [<the secret named huggingface-secret>], 'gpu': 'T4', 'timeout': 1800, 'min_containers': 0}

remote_prices(modal, ["1991 Hasbro Undertaker action figure, complete with urn",
                      "Replica Winged Eagle championship title belt, adult size"])
# [45.0, 389.0]   one lookup of pricer-service's Pricer class, one instance, one remote call per item

keep_warm(modal, 20)   # the Pricer instance, whose containers now stay awake for 1200 seconds after a call
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `run_remotely(modal, app, function, *args)`

Runs a function of an ephemeral Modal app once in the cloud, and returns its result. `modal` is the `modal` module,
`app` is a `modal.App`, and `function` is one of that app's functions, such as the lesson's `hello`. Turn Modal's
output on and start the app, each for the length of the call (as `with` blocks do), and while both are on, call the
function remotely once with `*args`, as the lesson's cells do. Return what it gives. Never run it locally: that runs
the function on this machine.

<details><summary>Hint</summary>

```python
with modal.enable_output():
    with app.run():
        return function.remote(*args)
```

</details>

In [ ]:
def run_remotely(modal, app, function, *args):
    # Write run_remotely here, as described above.
    raise NotImplementedError("run_remotely is not written yet")

In [ ]:
check("run_remotely")

### 2 · `service_options(modal, warm=0)`

Returns the keyword arguments for the pricer service's class decorator, as a dict, so that the service file can write
`@app.cls(**service_options(modal))`. The dict has exactly five keys:

- `"image"`: a Debian slim image with pip packages installed, at least `torch`, `transformers`, `bitsandbytes`,
  `accelerate` and `peft`, each given as its own name;
- `"secrets"`: a list holding the Modal secret named `"huggingface-secret"`;
- `"gpu"`: `"T4"`;
- `"timeout"`: `1800`, in seconds;
- `"min_containers"`: `warm`. A `warm` of 1 keeps one container always running: it answers at once, but it spends
  credits all the time.

<details><summary>Hint</summary>

```python
image = modal.Image.debian_slim().pip_install("torch", "transformers", ...)
secrets = [modal.Secret.from_name("huggingface-secret")]
```

</details>

In [ ]:
def service_options(modal, warm=0):
    # Write service_options here, as described above.
    raise NotImplementedError("service_options is not written yet")

In [ ]:
check("service_options")

### 3 · `remote_prices(modal, descriptions, app_name="pricer-service", class_name="Pricer")`

Prices each description with the deployed pricer class, and returns the prices as floats. Look up the deployed class
by its app's name and its own name, once, and make one instance of it. Then call the instance's `price` method
remotely, once per description, in order. Return a list with one float per description: the service can send back an
int, such as `0` when its reply holds no number. An empty list gives `[]`. The defaults are the names that the lesson
deploys.

<details><summary>Hint</summary>

```python
Pricer = modal.Cls.from_name(app_name, class_name)
pricer = Pricer()
pricer.price.remote(description)
```

</details>

In [ ]:
def remote_prices(modal, descriptions, app_name="pricer-service", class_name="Pricer"):
    # Write remote_prices here, as described above.
    raise NotImplementedError("remote_prices is not written yet")

In [ ]:
check("remote_prices")

### 4 · `keep_warm(modal, minutes, app_name="pricer-service", class_name="Pricer")`

Keeps the deployed pricer's containers awake for `minutes` after each call, and returns the instance. Look up and
instantiate the class as `remote_prices` does. Then update the instance's autoscaler so that its scale-down window is
the minutes in seconds: 20 minutes is `1200`. Pass the setting by name, since Modal's method takes nothing by
position. `keep_warm(modal, 2)` puts back the two minutes that the lesson returns to.

<details><summary>Hint</summary>

```python
pricer.update_autoscaler(scaledown_window=minutes * 60)
```

</details>

In [ ]:
def keep_warm(modal, minutes, app_name="pricer-service", class_name="Pricer"):
    # Write keep_warm here, as described above.
    raise NotImplementedError("keep_warm is not written yet")

In [ ]:
check("keep_warm")

## Check everything

The cell below runs all 7 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 8, Day 1 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Steps 1 to 3 use the pricer you deployed in the lesson. Modal bills the seconds its containers run, and the first call
can take a minute or more while a GPU container wakes up. Step 4 is optional: it builds and deploys a GPU service of
your own, and its first build can take up to 10 minutes.

1. Load your keys with `load_dotenv(override=True)` and `import modal`. Put the course's `week8` folder on the path
   with `sys.path.insert(0, str(ROOT / "week8"))`, import `app` and `hello_europe` from the lesson's `hello.py`, and
   print `run_remotely(modal, app, hello_europe)`.
2. Write a list `MERCH` of three pieces of wrestling merchandise, such as an Undertaker figure, a replica title belt
   and a WrestleMania III program, and print `remote_prices(modal, MERCH)`. It calls the `pricer-service` you deployed
   in the lesson.
3. Keep the service warm with `keep_warm(modal, 20)` while you price a few more items, then put it back with
   `keep_warm(modal, 2)`.
4. Optional: deploy a pricer of your own. Write it in a `%%writefile /tmp/merch_pricer.py` cell: an app named
   `merch-pricer`, the image and secret as in `service_options`, and a class `MerchPricer` under `@app.cls(...)`, with
   a `@modal.enter()` method that loads the fine-tuned model once and a `@modal.method()` `price`, as the lesson's
   `pricer_service2.py` does. Deploy it with `!uv run modal deploy /tmp/merch_pricer.py`, then price `MERCH` again
   with `app_name="merch-pricer", class_name="MerchPricer"`.

<details><summary>Stuck? One way to write it</summary>

```python
import sys

import modal
from dotenv import load_dotenv

load_dotenv(override=True)
sys.path.insert(0, str(ROOT / "week8"))  # the course's week8 folder
from hello import app, hello_europe

print(run_remotely(modal, app, hello_europe))
```

```python
MERCH = ["1991 Hasbro Undertaker action figure, complete with urn",
         "Replica Winged Eagle championship title belt, adult size, zinc plates on leather",
         "WrestleMania III event program, 1987, Pontiac Silverdome"]
print(remote_prices(modal, MERCH))  # the first call wakes the container up

keep_warm(modal, 20)  # warm for 20 minutes while you try more items
print(remote_prices(modal, ["Stone Cold Steve Austin action figure, Elite series, with vest and replica belt"]))
keep_warm(modal, 2)  # back to 2 minutes, so it stops spending
```

```python
%%writefile /tmp/merch_pricer.py
import modal

app = modal.App("merch-pricer")
image = modal.Image.debian_slim().pip_install("torch", "transformers", "bitsandbytes", "accelerate", "peft")
secrets = [modal.Secret.from_name("huggingface-secret")]
BASE_MODEL = "meta-llama/Llama-3.2-3B"
FINETUNED_MODEL = "ed-donner/price-2025-11-28_18.47.07"  # or your own adapter from Week 7
REVISION = "b19c8bfea3b6ff62237fbb0a8da9779fc12cefbd"  # None for your own adapter (or one of its commit ids)


@app.cls(image=image, secrets=secrets, gpu="T4", timeout=1800, min_containers=0)
class MerchPricer:
    @modal.enter()
    def setup(self):
        import torch
        from peft import PeftModel
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                          bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        self.tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant_config, device_map="auto")
        self.model = PeftModel.from_pretrained(base, FINETUNED_MODEL, revision=REVISION)

    @modal.method()
    def price(self, description: str) -> float:
        import re

        import torch
        from transformers import set_seed

        set_seed(42)  # as pricer_service2.py does, so an item gets the same price on every call
        prompt = f"What does this cost to the nearest dollar?\n\n{description}\n\nPrice is $"
        inputs = self.tokenizer.encode(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = self.model.generate(inputs, max_new_tokens=5)
        reply = self.tokenizer.decode(outputs[0]).split("Price is $")[-1].replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
        return float(match.group()) if match else 0.0
```

```python
!uv run modal deploy /tmp/merch_pricer.py
print(remote_prices(modal, MERCH, app_name="merch-pricer", class_name="MerchPricer"))
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.